# QuantLLMBot Phase 4 — Qwen2.5-14B QLoRA (Colab)

Runs the repo pipeline: preprocess → finetune → evaluate.

**Runtime:** GPU — **A100 40GB** recommended (L4 24GB works; T4 16GB: set fp16 and `max_seq_length` 2048 in `config.py`).

Data: `model_training/knowledge/stage_0{1,2,3,4}_*.jsonl` (115 train / 10 test).
Output: LoRA adapter in `model_training/outputs/lora_weights/`.

In [ ]:
# 1. GPU check
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Get the repo — Option A: clone (private repo needs a token)
# from getpass import getpass
# token = getpass('GitHub token: ')
# !git clone --depth 1 --branch QwenUpdate-version https://{token}@github.com/YOUR_USER/QuantLLMBot.git /content/QuantLLMBot

# Option B: upload a zip of the repo (scripts/ + model_training/knowledge/ is enough)
# from google.colab import files
# up = files.upload()  # upload QuantLLMBot.zip
# !unzip -q QuantLLMBot.zip -d /content/QuantLLMBot

import os
os.environ['QUANTLLM_ROOT'] = '/content/QuantLLMBot'
%cd /content/QuantLLMBot/scripts

In [ ]:
# 3. Install dependencies
!pip install -q -r requirements.txt

In [ ]:
# 4. Pre-flight checks
!python pre_training_checklist.py

In [ ]:
# 5. Preprocess (builds instruction-response pairs, 115 rows)
!python 01_preprocess.py

In [ ]:
# 6. Fine-tune Qwen2.5-14B with QLoRA (~15 GB download on first run)
!python 02_finetune.py

In [ ]:
# 7. Evaluate on the 10 held-out examples
!python 03_evaluate.py
!cat /content/QuantLLMBot/model_training/outputs/evaluation_results.json

In [ ]:
# 8. Download the adapter
!cd /content/QuantLLMBot/model_training/outputs && zip -qr qwen14b_lora_weights.zip lora_weights
from google.colab import files
files.download('/content/QuantLLMBot/model_training/outputs/qwen14b_lora_weights.zip')